# simulation 2 Mesh Series Analysis

This notebook analyzes a time series of OBJ meshes, calculates vertex-wise height changes over time, and extracts 4D Objects of Change (4D-OBCs).

**Dataset:** Simulation OBJ meshes
- **Location:** `C:\rsa\research_proj\blender_project\simulation_test2\data\sceneparts\simulation_test2`
- **Format:** OBJ files

**Assumption:** All meshes share the same vertex order and topology (e.g. they were produced from the same template / re-meshing pipeline), so vertex `i` in every epoch refers to the same physical location on the reference surface.

**Workflow:**
1. Load OBJ meshes as a time series of `py4dgeo.Epoch` objects (vertices used as the point cloud).
2. Optionally voxel-downsample the reference vertices and use the resulting subset as "corepoints" (the same indices are reused for every target mesh, preserving vertex correspondence).
3. For each target epoch, compute the **signed Z displacement** of every corepoint relative to its position in the reference mesh: `dz = z_target - z_reference`. Positive = above the reference surface, negative = below.
4. Build a `SpatiotemporalAnalysis` object with the signed displacements.
5. Run a custom 4D-OBC region-growing algorithm (`LinearChangeSeeds`, identical to `sand_dune_analysis.ipynb`).
6. Visualize the results.


In [ ]:
%load_ext autoreload
%autoreload 2

In [ ]:
import os
from datetime import datetime, timedelta

import numpy as np
import matplotlib.pyplot as plt
import trimesh
from tqdm import tqdm

import py4dgeo
# Import the new helper functions
from py4dgeo.data_loader import read_obj_and_assign_timestamps
from py4dgeo.segmentation import RegionGrowingAlgorithm

## 1. Configuration

In [ ]:
# The folder containing the .obj mesh files
data_path = r'C:\rsa\research_proj\blender_project\simulation_test2\data\sceneparts\simulation_test2'
output_path = os.path.join(os.getcwd(), 'simulation2_reference.zip')

# Timestamp of the reference mesh, same as the other sand dune notebook
reference_timestamp = datetime(2020, 1, 6, 0, 0, 0)

# Define the start time and increment for assigning timestamps
start_timestamp = datetime(2020, 1, 1, 0, 0, 0)
time_increment = timedelta(days=1)

vertex_voxel_size = 1

## 2. Load Mesh Epochs

In [ ]:
# Use the new helper function to read the OBJ files as epochs
# The vertices of each mesh are treated as the point cloud
epochs = read_obj_and_assign_timestamps(data_path, start_timestamp, time_increment)

if not epochs:
    raise RuntimeError(f"No epochs loaded. Check the data_path: {data_path}")

print(f"Loaded {len(epochs)} mesh epochs")
print(f"Date range: {epochs[0].timestamp} -> {epochs[-1].timestamp}")

# Set the reference epoch and the epochs to compare against
reference_epoch, other_epochs = py4dgeo.extract_reference_and_others(epochs, reference_timestamp)

# The corepoints are (a subset of) the vertices of the reference mesh.
# Because we rely on vertex-to-vertex correspondence across epochs, we must pick
# a fixed set of vertex INDICES on the reference mesh and reuse them on every
# target mesh - this is what `vertex_indices` is for.
all_ref_vertices = reference_epoch.cloud

if vertex_voxel_size is not None:
    # Voxel-based downsampling: keep one vertex per occupied voxel.
    # We hash each vertex into an integer voxel key, then keep the first
    # vertex falling into each unique voxel.
    voxel_keys = np.floor(all_ref_vertices / vertex_voxel_size).astype(np.int64)
    _, first_idx = np.unique(voxel_keys, axis=0, return_index=True)
    vertex_indices = np.sort(first_idx)
    print(f"Voxel downsampling (size={vertex_voxel_size} m): "
          f"{len(all_ref_vertices):,} -> {len(vertex_indices):,} vertices")
else:
    vertex_indices = np.arange(len(all_ref_vertices))
    print(f"No downsampling: using all {len(vertex_indices):,} vertices")

corepoints = all_ref_vertices[vertex_indices]

print(f"Reference: {reference_epoch.timestamp} with {len(corepoints):,} corepoints")
print(f"Other epochs to process: {len(other_epochs)}")


In [ ]:
corepoints

## 3. Calculate Signed Vertex Displacements Over Time

For every corepoint (a vertex index selected on the reference mesh) and every target epoch, we compute the **signed Z-axis displacement** of the corresponding vertex on the target mesh, using the reference vertex as the baseline:

$$
\text{dz}_{i,t} \;=\; z^{\text{target}}_{i,t} \;-\; z^{\text{reference}}_{i}
$$

**Consequences:**
- A positive value means the surface has risen above the reference (e.g. sediment deposition).
- A negative value means the surface has dropped below the reference (e.g. erosion).
- The reference timestamp itself would have `dz = 0` everywhere (it is excluded from `other_epochs`).

This only works because all OBJ meshes share the same vertex order: vertex `vertex_indices[k]` in the reference mesh and in the target mesh refer to the same physical point of the modelled surface.


In [ ]:
# For every target mesh, compute the SIGNED Z-displacement (target - reference)
# at the selected `vertex_indices`. Positive values = above reference (deposition),
# negative values = below reference (erosion).

# Initialize the array. Shape: (num_corepoints, num_other_epochs)
distances = np.full((len(corepoints), len(other_epochs)), np.nan, dtype=np.float64)

# All OBJ files sorted alphabetically (assumed to be the same ordering used by
# `read_obj_epochs_from_folder` to assign timestamps).
obj_files_sorted = sorted(f for f in os.listdir(data_path) if f.lower().endswith('.obj'))

# Load the reference mesh once and extract the reference Z coordinates at the
# corepoint vertex indices - this is the baseline.
ref_index_in_all_epochs = epochs.index(reference_epoch)
ref_mesh_path = os.path.join(data_path, obj_files_sorted[ref_index_in_all_epochs])
ref_mesh = trimesh.load(ref_mesh_path, force='mesh', process=False)
ref_z = ref_mesh.vertices[vertex_indices, 2]

for i, target_epoch in enumerate(tqdm(other_epochs, desc="Calculating signed Z displacements")):
    target_index_in_all_epochs = epochs.index(target_epoch)
    target_mesh_path = os.path.join(data_path, obj_files_sorted[target_index_in_all_epochs])
    target_mesh = trimesh.load(target_mesh_path, force='mesh', process=False)

    # Vertex correspondence requires identical vertex counts.
    if len(target_mesh.vertices) != len(ref_mesh.vertices):
        print(f"Warning: Mesh vertex count for {target_epoch.timestamp} "
              f"({len(target_mesh.vertices)}) != reference "
              f"({len(ref_mesh.vertices)}). Skipping.")
        continue

    # Signed Z difference at the selected corepoint indices.
    target_z = target_mesh.vertices[vertex_indices, 2]
    distances[:, i] = target_z - ref_z

print(f"Signed displacement matrix shape: {distances.shape}")
print(f"  min = {np.nanmin(distances):.4f} m   max = {np.nanmax(distances):.4f} m")
print(f"  mean = {np.nanmean(distances):.4f} m   median = {np.nanmedian(distances):.4f} m")

## 4. Spatiotemporal Analysis and 4D-OBC

In [ ]:
# 4D-OBC parameters 
obc_neighborhood_radius = 1
obc_min_segments = 10
obc_minperiod = 3
obc_height_threshold = 0.05
obc_thresholds = [0.5, 0.6, 0.7, 0.8, 0.9]

In [ ]:
# Create a fresh analysis object for the mesh-derived reference distances.
# force=True is important because distances and timestamps can only be written once.
analysis = py4dgeo.SpatiotemporalAnalysis(output_path, force=True)

# Populate it with our data.
analysis.reference_epoch = reference_epoch
analysis.corepoints = corepoints
analysis.timedeltas = [e.timestamp - reference_epoch.timestamp for e in other_epochs]
analysis.distances = distances

# We don't have a separate M3C2 algorithm, so we leave it as None.
analysis.m3c2 = None

# We also don't have uncertainties in this workflow.
analysis.uncertainties = np.full(
    distances.shape,
    np.nan,
    dtype=[
        ("lodetection", "<f8"),
        ("spread1", "<f8"),
        ("num_samples1", "<i8"),
        ("spread2", "<f8"),
        ("num_samples2", "<i8"),
    ],
)

print(f"Stored {len(analysis.timedeltas)} target timestamps")
print(f"Smoothed distances shape: {analysis.distances.shape}")

In [ ]:
indices = np.nonzero(analysis.distances)[0]

np.unique(indices)

In [ ]:
# Run the 4D-OBC algorithm
algo = RegionGrowingAlgorithm(
    neighborhood_radius=obc_neighborhood_radius,
    min_segments=obc_min_segments,
    minperiod=obc_minperiod,
    height_threshold=obc_height_threshold,
    thresholds=obc_thresholds,
    seed_subsampling=1, # Subsample seeds to speed up the process
)

analysis.invalidate_results(seeds=True, objects=True, smoothed_distances=False)
objects = algo.run(analysis)
print(f"Extracted {len(objects)} 4D-OBCs from {len(analysis.seeds)} seeds")

In [ ]:
analysis.objects

## 5. Visualize Results

In [ ]:
# Plot the time series for a selected corepoint
if analysis.smoothed_distances is not None and analysis.smoothed_distances.shape[1] > 0 and len(corepoints) > 0:
    timestamps = [e.timestamp for e in other_epochs]

    cp_idx_sel = 3420 
    ts = analysis.smoothed_distances[cp_idx_sel]
    cp_coords = corepoints[cp_idx_sel]

    plt.figure(figsize=(12, 5))
    plt.plot(timestamps, ts, c='black', ls='--', lw=0.7, label='Z-Displacement Time Series')

    # Highlight the segments that were identified as seeds for this corepoint
    for sid, s in enumerate(s for s in analysis.seeds if s.index == cp_idx_sel):
        plt.plot(
            timestamps[s.start_epoch:s.end_epoch + 1],
            ts[s.start_epoch:s.end_epoch + 1],
            lw=2, label=f'seed {sid}: epochs {s.start_epoch}-{s.end_epoch}'
        )
        
    plt.xlabel('Time')
    plt.ylabel('Z-Displacement [m]')
    plt.title(f'Time Series for Corepoint Index {cp_idx_sel} (Coords: {cp_coords[0]:.2f}, {cp_coords[1]:.2f}, {cp_coords[2]:.2f})')
    plt.xticks(rotation=45)
    plt.grid(alpha=0.3)
    plt.legend()
    plt.tight_layout()
    plt.show()

In [ ]:
# plot the N-th object and its seed info
sel_object_idx = 1

if len(objects) > 0:
    sel_obj = analysis.objects[sel_object_idx]
    sel_seed = sel_obj.seed  # Use the seed stored on the object itself (NOT analysis.seeds[sel_object_idx])
    print(f"Object {sel_object_idx}: seed CP={sel_seed.index}, epochs {sel_seed.start_epoch}-{sel_seed.end_epoch}, size={len(sel_obj.indices)}")
    sel_obj.plot()

In [ ]:
# plot the N-th object and its seed info, with more details
from scipy.spatial import ConvexHull
from matplotlib.patches import Polygon
import matplotlib.colors as mcolors

sel_object_idx = 0 # object index to visualize

if len(objects) > 0:
    sel_object = analysis.objects[sel_object_idx]
    # get the seed from the object itself, NOT from analysis.seeds[i].
    sel_seed = sel_object.seed # seed index of the object
    seed_cp_idx = sel_seed.index # seed corepoint index

    fig, axs = plt.subplots(1, 2, figsize=(15, 5))
    ax1, ax2 = axs

    idxs = sel_object.indices
    epoch_of_interest = int(sel_object.end_epoch)
    magnitudes_of_interest = (analysis.smoothed_distances[:, epoch_of_interest] -
                              analysis.smoothed_distances[:, int(sel_object.start_epoch)])

    crange = 0.2
    cmap = plt.get_cmap('seismic_r').copy()
    norm = mcolors.CenteredNorm(halfrange=crange)
    cmapvals = norm(magnitudes_of_interest)

    for idx in idxs[::10]:
        ax1.plot(timestamps, analysis.smoothed_distances[idx],
                c=cmap(cmapvals[idx]), linewidth=0.5)
    ax1.plot(timestamps, analysis.smoothed_distances[seed_cp_idx],
            c='black', linewidth=1., label='Seed timeseries')
    ax1.axvspan(timestamps[sel_object.start_epoch], timestamps[sel_object.end_epoch],
               alpha=0.3, color='grey', label='4D-OBC timespan')
    ax1.legend()
    ax1.set_title('Time series of segmented 4D-OBC locations')
    ax1.set_xlabel('Date')
    ax1.set_ylabel('Distance [m]')
    ax1.grid(True, alpha=0.3)
    plt.setp(ax1.xaxis.get_majorticklabels(), rotation=45)

    cloud = analysis.corepoints.cloud
    subset_cloud = cloud[idxs, :2]

    d = ax2.scatter(cloud[:, 0], cloud[:, 1], c=magnitudes_of_interest,
                   cmap='seismic_r', vmin=-crange, vmax=crange, s=1)
    plt.colorbar(d, format='%.2f', label='Change magnitude [m]', ax=ax2)

    if len(subset_cloud) >= 3:
        hull = ConvexHull(subset_cloud)
        ax2.add_patch(Polygon(subset_cloud[hull.vertices, 0:2],
                             label='4D-OBC hull', fill=False, edgecolor='black'))

    ax2.scatter(cloud[seed_cp_idx, 0], cloud[seed_cp_idx, 1],
               marker='*', s=200, c='black',
               label=f'Seed (CP {seed_cp_idx})', zorder=5)

    ax2.set_title('Spatial distribution of 4D-OBC')
    ax2.set_xlabel('X [m]')
    ax2.set_ylabel('Y [m]')
    ax2.legend(loc='upper right')
    ax2.axis('equal')
    plt.tight_layout()
    plt.show()
else:
    print("No objects to visualize!")

In [ ]:
# Changemap: epoch with the largest absolute change relative to the reference epoch
import matplotlib.colors as mcolors

change_matrix = analysis.distances
if change_matrix is None or change_matrix.shape[1] == 0:
    raise ValueError("analysis.distances is empty. Run the distance calculation cells first.")

cloud = analysis.corepoints.cloud
if cloud.shape[0] != change_matrix.shape[0]:
    raise ValueError(
        f"Corepoint count ({cloud.shape[0]}) does not match distance rows ({change_matrix.shape[0]})."
    )

try:
    target_timestamps = [e.timestamp for e in other_epochs]
except NameError:
    target_timestamps = [reference_epoch.timestamp + dt for dt in analysis.timedeltas]

if len(target_timestamps) != change_matrix.shape[1]:
    target_timestamps = list(range(change_matrix.shape[1]))

with np.errstate(all="ignore"):
    epoch_strength = np.nanmax(np.abs(change_matrix), axis=0)

if not np.isfinite(epoch_strength).any():
    raise ValueError("No finite change values found in analysis.distances.")

max_change_epoch_idx = int(np.nanargmax(epoch_strength))
values = change_matrix[:, max_change_epoch_idx]
finite_values = values[np.isfinite(values)]
if finite_values.size == 0:
    raise ValueError("No finite change values found for the selected maximum-change epoch.")

# Display range for the color scale. Values outside this range are clipped to
# the end colors, so ~1 m terrain changes remain visually distinct even when
# a few places change by much more.
display_crange = 1.5
crange = min(display_crange, float(np.nanmax(np.abs(finite_values))))
if crange <= 0 or not np.isfinite(crange):
    crange = display_crange

timestamp = target_timestamps[max_change_epoch_idx]
target_label = timestamp.date() if hasattr(timestamp, "date") else timestamp
reference_label = reference_epoch.timestamp.date()
max_abs = float(np.nanmax(np.abs(values)))

fig, ax = plt.subplots(figsize=(8, 7), constrained_layout=True)
cmap = plt.get_cmap("seismic_r").copy()
cmap.set_bad("lightgrey")
norm = mcolors.TwoSlopeNorm(vmin=-crange, vcenter=0.0, vmax=crange)

sc = ax.scatter(
    cloud[:, 0],
    cloud[:, 1],
    c=values,
    cmap=cmap,
    norm=norm,
    s=2,
    linewidths=0,
)
fig.colorbar(
    sc,
    ax=ax,
    shrink=0.9,
    extend="both",
    label=f"Signed change vs reference [m], clipped to +/-{crange:.1f} m",
)

# ax.set_title(
    # f"Largest absolute change\n{target_label} vs {reference_label} | max |change| = {max_abs:.3f} m"
# )
ax.set_xlabel("X [m]")
ax.set_ylabel("Y [m]")
ax.axis("equal")
ax.grid(alpha=0.2)

print(
    f"Largest absolute change: epoch_idx={max_change_epoch_idx}, timestamp={timestamp}, "
    f"min={np.nanmin(values):.4f} m, max={np.nanmax(values):.4f} m, max_abs={max_abs:.4f} m"
)
print(f"Color display range clipped to +/-{crange:.2f} m; larger changes use the end colors.")
plt.show()

In [ ]:
# Changemap: final epoch relative to the reference epoch
import matplotlib.colors as mcolors

change_matrix = analysis.distances
if change_matrix is None or change_matrix.shape[1] == 0:
    raise ValueError("analysis.distances is empty. Run the distance calculation cells first.")

cloud = analysis.corepoints.cloud
if cloud.shape[0] != change_matrix.shape[0]:
    raise ValueError(
        f"Corepoint count ({cloud.shape[0]}) does not match distance rows ({change_matrix.shape[0]})."
    )

try:
    target_timestamps = [e.timestamp for e in other_epochs]
except NameError:
    target_timestamps = [reference_epoch.timestamp + dt for dt in analysis.timedeltas]

if len(target_timestamps) != change_matrix.shape[1]:
    target_timestamps = list(range(change_matrix.shape[1]))

last_epoch_idx = change_matrix.shape[1] - 1
values = change_matrix[:, last_epoch_idx]
finite_values = values[np.isfinite(values)]
if finite_values.size == 0:
    raise ValueError("No finite change values found for the final epoch.")

# Display range for the color scale. Values outside this range are clipped to
# the end colors, so ~1 m terrain changes remain visually distinct even when
# a few places change by much more.
display_crange = 1.5
crange = min(display_crange, float(np.nanmax(np.abs(finite_values))))
if crange <= 0 or not np.isfinite(crange):
    crange = display_crange

timestamp = target_timestamps[last_epoch_idx]
target_label = timestamp.date() if hasattr(timestamp, "date") else timestamp
reference_label = reference_epoch.timestamp.date()
max_abs = float(np.nanmax(np.abs(values)))

fig, ax = plt.subplots(figsize=(8, 7), constrained_layout=True)
cmap = plt.get_cmap("seismic_r").copy()
cmap.set_bad("lightgrey")
norm = mcolors.TwoSlopeNorm(vmin=-crange, vcenter=0.0, vmax=crange)

sc = ax.scatter(
    cloud[:, 0],
    cloud[:, 1],
    c=values,
    cmap=cmap,
    norm=norm,
    s=2,
    linewidths=0,
)
fig.colorbar(
    sc,
    ax=ax,
    shrink=0.9,
    extend="both",
    label=f"Signed change vs reference [m], clipped to +/-{crange:.1f} m",
)

ax.set_title(
    f"Final epoch\n{target_label} vs {reference_label} | max |change| = {max_abs:.3f} m"
)
ax.set_xlabel("X [m]")
ax.set_ylabel("Y [m]")
ax.axis("equal")
ax.grid(alpha=0.2)

print(
    f"Final epoch: epoch_idx={last_epoch_idx}, timestamp={timestamp}, "
    f"min={np.nanmin(values):.4f} m, max={np.nanmax(values):.4f} m, max_abs={max_abs:.4f} m"
)
print(f"Color display range clipped to +/-{crange:.2f} m; larger changes use the end colors.")
plt.show()